# Master Regeneration Pipeline — SMID & LJ

**Single parameterized script for both aircraft segments.**

**Input**: `lj_smid_us_revenue_clean_24_25_metro.parquet`  
**Output**: `{segment}_comprehensive.csv` (smid or lj)

Final dataset includes **seasonality multiplier** (not raw index).

---

## Configuration

Select which segment to regenerate:

In [66]:
# ============================================
# SELECT SEGMENT HERE
# ============================================
SEGMENT = "SMID"  # Options: "SMID" or "LJ"
# ============================================

# Segment configurations
CONFIGS = {
    "SMID": {
        "segment_name": "Super Midsize Jet",
        "models": None,  # All models in segment
        "min_hours": 2.5,
        "min_missions": 30,
        "cluster_labels": {
            0: "winter-spring-peak",
            1: "summer-peak",
            2: "multi-peak-directional",
            3: "core-utilization"
        },
        "output_file": "smid_comprehensive.csv",
        "si_multiplier_map": {
            "< 0.75x": 0.75,
            "0.75x to < 0.90x": 0.90,
            "0.90x to < 1.10x": 1.00,
            "1.10x to < 1.30x": 1.15,
            "1.30x to < 1.60x": 1.30,
            ">= 1.60x": 1.50
        }
    },
    "LJ": {
        "segment_name": "Light Jet",
        "models": ['Phenom 300', 'Citation CJ', 'Hawker 400', 'Beechjet', 'Encore', 'Ultra'],
        "min_hours": 1.5,
        "min_missions": 20,
        "cluster_labels": {
            0: "winter-spring-peak",
            1: "summer-peak",
            2: "core-utilization",
            3: "multi-peak-directional"
        },
        "output_file": "lj_comprehensive.csv",
        "si_multiplier_map": {
            "< 0.75x": 0.75,
            "0.75x to < 0.90x": 0.90,
            "0.90x to < 1.10x": 1.00,
            "1.10x to < 1.30x": 1.15,
            "1.30x to < 1.60x": 1.30,
            ">= 1.60x": 1.50
        }
    }
}

config = CONFIGS[SEGMENT]

print(f"🎯 Configuration: {SEGMENT}")
print(f"   Segment: {config['segment_name']}")
print(f"   Min hours: {config['min_hours']}")
print(f"   Min missions/yr: {config['min_missions']}")
print(f"   Output: {config['output_file']}")
print(f"   Models: {config['models'] if config['models'] else 'All'}")
print(f"\n📊 SI → Multiplier Mapping:")
for si_slab, mult in config['si_multiplier_map'].items():
    print(f"   {si_slab:20} → {mult}")

🎯 Configuration: SMID
   Segment: Super Midsize Jet
   Min hours: 2.5
   Min missions/yr: 30
   Output: smid_comprehensive.csv
   Models: All

📊 SI → Multiplier Mapping:
   < 0.75x              → 0.75
   0.75x to < 0.90x     → 0.9
   0.90x to < 1.10x     → 1.0
   1.10x to < 1.30x     → 1.15
   1.30x to < 1.60x     → 1.3
   >= 1.60x             → 1.5


---

## Pipeline

In [67]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

print("✓ Dependencies loaded.")

✓ Dependencies loaded.


### STEP 1: LOAD & STANDARDIZE

In [68]:
RAW_PARQUET = "gs://agntworks-data-dev/wheelsup/processed/wingx/lj_smid_us_revenue_clean_24_25_metro_local_TimeZone.parquet"

df = pd.read_parquet(RAW_PARQUET)
df['FlightDate_local'] = pd.to_datetime(df['FlightDate_local'])
df = df.rename(columns={'FromCluster': 'FromMetro', 'ToCluster': 'ToMetro'})
print(f"✓ Loaded {len(df):,} records")
print(f"  Date range: {df['FlightDate_local'].min().date()} to {df['FlightDate_local'].max().date()}")

✓ Loaded 2,140,706 records
  Date range: 2023-12-31 to 2025-12-31


### STEP 2: FILTER TO SEGMENT INTER-METRO

In [69]:
# Build filter mask
mask = (
    (df['aircraft_segment'] == config['segment_name']) &
    (df['FromMetro'] != 'OTHER_METRO') &
    (df['ToMetro'] != 'OTHER_METRO') &
    (df['FromMetro'] != df['ToMetro']) &
    (df['Hours'] >= config['min_hours'])
)

# Add model filter if specified
if config['models']:
    mask = mask & (df['aircraft_model'].isin(config['models']))

segment_lh = df[mask].copy()
segment_lh['corridor'] = segment_lh['FromMetro'] + " -> " + segment_lh['ToMetro']

print(f"✓ Filtered to {len(segment_lh):,} {SEGMENT} inter-metro records")
print(f"  Unique corridors: {segment_lh['corridor'].nunique():,}")

intra_metro_check = (segment_lh['FromMetro'] == segment_lh['ToMetro']).sum()
print(f"  Intra-metro: {intra_metro_check} (expect 0)")

✓ Filtered to 211,133 SMID inter-metro records
  Unique corridors: 447
  Intra-metro: 0 (expect 0)


### STEP 3: SEASONAL AGGREGATION

In [70]:
seasonal_counts = (
    segment_lh
    .groupby(["corridor", "month_local"])
    .size()
    .unstack(fill_value=0)
)

seasonal_counts = seasonal_counts.reindex(columns=range(1, 13), fill_value=0)

annual_missions = seasonal_counts.sum(axis=1)
relevant_corridors = annual_missions > config['min_missions']
seasonal_counts_relevant = seasonal_counts.loc[relevant_corridors].copy()

print(f"✓ Seasonal aggregation complete")
print(f"  Corridors with >{config['min_missions']} missions/year: {len(seasonal_counts_relevant)}")

✓ Seasonal aggregation complete
  Corridors with >30 missions/year: 289


### STEP 4: KMEANS CLUSTERING & SEASONALITY INDEX

In [71]:
monthly_density = seasonal_counts_relevant.div(
    seasonal_counts_relevant.sum(axis=1),
    axis=0
)

seasonality_index = monthly_density / (1 / 12)

km = KMeans(n_clusters=4, random_state=42, n_init=50)
cluster_ids = km.fit_predict(monthly_density)

X_discovery = monthly_density.copy()
X_discovery["cluster_id"] = cluster_ids

for m in range(1, 13):
    X_discovery[f"seasonality_index_m{m}"] = seasonality_index[m]

print(f"✓ KMeans clustering complete (k=4, random_state=42)")
print(f"  Seasonality index computed")

unique, counts = np.unique(cluster_ids, return_counts=True)
print(f"\n  Cluster distribution:")
for cid, cnt in zip(unique, counts):
    label = config['cluster_labels'][cid]
    pct = cnt / len(cluster_ids) * 100
    print(f"    {cid} ({label}): {cnt:,} corridors ({pct:.1f}%)")

✓ KMeans clustering complete (k=4, random_state=42)
  Seasonality index computed

  Cluster distribution:
    0 (winter-spring-peak): 61 corridors (21.1%)
    1 (summer-peak): 70 corridors (24.2%)
    2 (multi-peak-directional): 117 corridors (40.5%)
    3 (core-utilization): 41 corridors (14.2%)


### STEP 5: BUILD & EXPORT COMPREHENSIVE

In [72]:
MONTH_NAMES = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
               7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
MONTH_NUM   = {v:k for k,v in MONTH_NAMES.items()}

# Long format
density_cols = list(range(1, 13))
base = X_discovery[density_cols + ["cluster_id"]].copy().reset_index()

for m in density_cols:
    base[m] = (base[m] * 100).round(2)

base = base.rename(columns=MONTH_NAMES)
base["cluster_label"] = base["cluster_id"].map(config['cluster_labels'])

comp = base.melt(
    id_vars=["corridor", "cluster_id", "cluster_label"],
    value_vars=list(MONTH_NAMES.values()),
    var_name="month", value_name="density_pct"
)
comp["month_num"] = comp["month"].map(MONTH_NUM)

# Seasonality index slab (for mapping to multiplier)
bins_si   = [0, 0.75, 0.90, 1.10, 1.30, 1.60, float("inf")]
labels_si = ["< 0.75x", "0.75x to < 0.90x", "0.90x to < 1.10x",
             "1.10x to < 1.30x", "1.30x to < 1.60x", ">= 1.60x"]

# Seasonality index
si_long = (X_discovery[[f"seasonality_index_m{m}" for m in range(1,13)]].reset_index()
    .melt(id_vars="corridor", var_name="si_col", value_name="seasonality_index"))
si_long["month_num"] = si_long["si_col"].map({f"seasonality_index_m{m}": m for m in range(1,13)})
comp = comp.merge(si_long[["corridor","month_num","seasonality_index"]],
                  on=["corridor","month_num"], how="left")

# Bin seasonality index into slab
comp["si_slab"] = pd.cut(comp["seasonality_index"], bins=bins_si, labels=labels_si, right=False)

# Map slab to multiplier
comp["multiplier"] = comp["si_slab"].map(config['si_multiplier_map'])

print(f"✓ Seasonality index binned and multiplier mapped")

✓ Seasonality index binned and multiplier mapped


In [75]:
# Density pct slab (for reference)
bins_dp   = [0, 4, 6, 9, 12, 15, float("inf")]
labels_dp = ["< 4%", "4% to < 6%", "6% to < 9%", "9% to < 12%", "12% to 15%", "> 15%"]

comp["density_slab"] = pd.cut(comp["density_pct"], bins=bins_dp, labels=labels_dp, right=False)

# Round seasonality_index to 4 decimals
comp["seasonality_index"] = comp["seasonality_index"].round(4)

# Final columns: cluster_label | corridor | month | density_pct | density_slab | seasonality_index | multiplier | si_slab
comp = comp[[
    "cluster_label","corridor","month","density_pct","density_slab","seasonality_index","multiplier","si_slab"
]].sort_values(["cluster_label","corridor","month"]).reset_index(drop=True)

print(f"✓ Comprehensive dataset built: {len(comp):,} rows × {len(comp.columns)} columns")

✓ Comprehensive dataset built: 3,468 rows × 2 columns


In [76]:
output_path = config['output_file']
comp.to_csv(output_path, index=False)

print(f"\n✅ EXPORTED: {output_path}")
print(f"\n📊 Summary:")
print(f"  Rows: {len(comp):,}")
print(f"  Columns: {comp.columns.tolist()}")
print(f"  Unique corridors: {comp['corridor'].nunique()}")

print(f"\n📍 Cluster breakdown:")
cluster_summary = comp.groupby('cluster_label')['corridor'].nunique().sort_values(ascending=False)
for cluster, count in cluster_summary.items():
    print(f"  {cluster}: {count} corridors")

print(f"\n🛡️ Validation:")
print(f"  Null values: {comp.isnull().sum().sum()}")
print(f"  Density % range: {comp['density_pct'].min():.2f}% - {comp['density_pct'].max():.2f}%")
print(f"  Seasonality index range: {comp['seasonality_index'].min():.4f}x - {comp['seasonality_index'].max():.4f}x")
print(f"  Multiplier range: {comp['multiplier'].min()} - {comp['multiplier'].max()}")

print(f"\n📋 Multiplier distribution:")
mult_dist = comp['multiplier'].value_counts().sort_index()
for mult, count in mult_dist.items():
    pct = count / len(comp) * 100
    print(f"  {mult}: {count:,} rows ({pct:.1f}%)")

print(f"\n📋 Sample rows:")
display(comp.head(15))


✅ EXPORTED: smid_comprehensive.csv

📊 Summary:
  Rows: 3,468
  Columns: ['cluster_label', 'corridor']
  Unique corridors: 289

📍 Cluster breakdown:
  multi-peak-directional: 117 corridors
  summer-peak: 70 corridors
  winter-spring-peak: 61 corridors
  core-utilization: 41 corridors

🛡️ Validation:
  Null values: 0

📋 Sample rows:


,cluster_label,corridor
0,core-utilization,Atlanta -> Denver
1,core-utilization,Atlanta -> Denver
2,core-utilization,Atlanta -> Denver
3,core-utilization,Atlanta -> Denver
4,core-utilization,Atlanta -> Denver
5,core-utilization,Atlanta -> Denver
6,core-utilization,Atlanta -> Denver
7,core-utilization,Atlanta -> Denver
8,core-utilization,Atlanta -> Denver
9,core-utilization,Atlanta -> Denver
